# Hoja de Trabajo 7 — CC2017 Modelación y Simulación
**Grupo 4** · Hospital Regional del Altiplano (camas de UCI)

## Task 1
### Task 1.1 — Desempeño teórico de las dos variantes

In [1]:
import math

# ---------- Parámetros del grupo ----------
g   = 4                              # número de grupo
mu  = 0.25                           # tasa de servicio [1/día]  (estancia media 1/mu = 4 días)
lam = 3.0 + 0.2 * ((g - 1) % 5)      # tasa de llegadas [pacientes/día]
c   = 17 + ((g - 1) % 3)             # número de camas
a   = lam / mu                       # carga ofrecida (Erlangs)
rho = a / c                          # utilización por cama
k_off = math.ceil(c - a) + 3         # camas fuera de servicio en el cierre parcial
T_b = 7                              # duración del cierre [días]

print(f"Grupo g = {g}")
print(f"λ     = {lam:.2f} pacientes/día")
print(f"c     = {c} camas")
print(f"a     = λ/μ = {a:.2f} Erlangs")
print(f"ρ     = a/c = {rho:.6f}")
print(f"k_off = ceil(c - a) + 3 = ceil({c - a:.2f}) + 3 = {k_off} camas   (T_b = {T_b} días)")
assert rho < 1, "La variante con espera necesita ρ < 1 para ser estable"


Grupo g = 4
λ     = 3.60 pacientes/día
c     = 17 camas
a     = λ/μ = 14.40 Erlangs
ρ     = a/c = 0.847059
k_off = ceil(c - a) + 3 = ceil(2.60) + 3 = 6 camas   (T_b = 7 días)


In [2]:
def erlang_b(c, a):
    # Recurrencia: B(k) = a*B(k-1) / (k + a*B(k-1)),  B(0) = 1
    B = 1.0
    for k in range(1, c + 1):
        B = a * B / (k + a * B)
    return B

def erlang_b_directa(c, a):
    # Fórmula directa: (a^c/c!) / sum_{k=0}^{c} a^k/k!   (solo para validar la recurrencia)
    num = a**c / math.factorial(c)
    den = sum(a**k / math.factorial(k) for k in range(c + 1))
    return num / den

def erlang_c(c, a):
    # C(c,a) = B / (1 - rho*(1 - B)),  requiere rho = a/c < 1
    rho = a / c
    B = erlang_b(c, a)
    return B / (1 - rho * (1 - B))

def erlang_c_directa(c, a):
    # Fórmula directa de Erlang C (solo para validar)
    rho = a / c
    cola = a**c / math.factorial(c) / (1 - rho)
    return cola / (sum(a**k / math.factorial(k) for k in range(c)) + cola)

# Validación cruzada de las dos formas de cada fórmula
print(f"Erlang B  recurrencia = {erlang_b(c, a):.10f} | directa = {erlang_b_directa(c, a):.10f}")
print(f"Erlang C  vía B       = {erlang_c(c, a):.10f} | directa = {erlang_c_directa(c, a):.10f}")


Erlang B  recurrencia = 0.0967215320 | directa = 0.0967215320
Erlang C  vía B       = 0.4118087900 | directa = 0.4118087900


#### (a) Variante con espera — M/M/c

In [3]:
# ---------- Variante CON espera (M/M/c) ----------
C_ca = erlang_c(c, a)                          # P(esperar) = C(c,a)
Lq   = C_ca * rho / (1 - rho)                  # Lq = C * rho/(1-rho)
Wq_d = Lq / lam                                # Wq = Lq / λ            [días]
W_d  = Wq_d + 1 / mu                           # W  = Wq + 1/μ          [días]
L    = Lq + a                                  # L  = Lq + a  (a = camas ocupadas en promedio)

Wq_h = Wq_d * 24                               # a horas
W_h  = W_d * 24

# Verificación independiente de Wq con la forma cerrada  Wq = C / (c*μ - λ)
Wq_alt_d = C_ca / (c * mu - lam)

print("=== Variante con espera (M/M/c) ===")
print(f"a      = {a:.4f}")
print(f"ρ      = {rho:.6f}")
print(f"C(c,a) = {C_ca:.6f}   (P[esperar] ≈ {C_ca*100:.2f} %)")
print(f"Lq     = {Lq:.6f} pacientes")
print(f"Wq     = {Wq_d:.6f} días = {Wq_h:.4f} horas")
print(f"W      = {W_d:.6f} días = {W_h:.4f} horas")
print(f"L      = {L:.6f} pacientes")
print(f"(check Wq = C/(cμ-λ): {Wq_alt_d:.6f} días)")


=== Variante con espera (M/M/c) ===
a      = 14.4000
ρ      = 0.847059
C(c,a) = 0.411809   (P[esperar] ≈ 41.18 %)
Lq     = 2.280787 pacientes
Wq     = 0.633552 días = 15.2052 horas
W      = 4.633552 días = 111.2052 horas
L      = 16.680787 pacientes
(check Wq = C/(cμ-λ): 0.633552 días)


#### (b) Variante sin espera — M/M/c/c (Erlang B / sistema de pérdida)

In [4]:
# ---------- Variante SIN espera (M/M/c/c) ----------
B_ca    = erlang_b(c, a)                       # prob. de bloqueo B(c,a)
lam_ef  = lam * (1 - B_ca)                     # tasa efectiva de ingresos: solo entran los no bloqueados
L_sin   = lam_ef / mu                          # camas ocupadas promedio = λ_ef/μ = a(1-B)
desvio  = lam * B_ca                           # pacientes desviados por día
W_sin_d = 1 / mu                               # no hay espera: W = 1/μ
W_sin_h = W_sin_d * 24

print("=== Variante sin espera (M/M/c/c) ===")
print(f"B(c,a)          = {B_ca:.6f}   ({B_ca*100:.3f} % de pacientes desviados)")
print(f"λ_ef            = {lam_ef:.6f} pacientes/día   (λ = {lam:.2f})")
print(f"Camas ocupadas  = {L_sin:.6f}   (de {c}; ocupación por cama = {L_sin/c:.4f})")
print(f"Desvíos         = {desvio:.6f} pacientes/día")
print(f"W = 1/μ         = {W_sin_d:.2f} días = {W_sin_h:.1f} horas; Wq = 0")


=== Variante sin espera (M/M/c/c) ===
B(c,a)          = 0.096722   (9.672 % de pacientes desviados)
λ_ef            = 3.251802 pacientes/día   (λ = 3.60)
Camas ocupadas  = 13.007210   (de 17; ocupación por cama = 0.7651)
Desvíos         = 0.348198 pacientes/día
W = 1/μ         = 4.00 días = 96.0 horas; Wq = 0


#### (c) Consistencia con la ley de Little

$L = \lambda_{\text{ef}}\,W$ debe cumplirse **dentro de cada variante** (con $\lambda_{\text{ef}}$ la tasa a la que realmente entran pacientes al sistema). Además, la relación entre variantes y las identidades internas:

- **Con espera:** $\lambda_{\text{ef}} = \lambda$ (nadie se pierde), por lo que $L_q = \lambda W_q$ y $L = \lambda W$.
- **Sin espera:** $L = \lambda_{\text{ef}}\,W = \lambda(1-B)\cdot\dfrac{1}{\mu}$, y $L_q = \lambda_{\text{ef}} W_q = 0$ trivialmente.

In [5]:
# ---------- Verificación con la ley de Little: L = λ_ef * W ----------
# Variante con espera (λ_ef = λ)
little_Lq = lam * Wq_d            # debe igualar Lq
little_L  = lam * W_d             # debe igualar L

# Variante sin espera (λ_ef = λ(1-B))
little_L_sin = lam_ef * W_sin_d   # debe igualar L_sin

print("--- Variante con espera ---")
print(f"Lq      = {Lq:.8f}   | λ·Wq = {little_Lq:.8f}   | dif = {abs(Lq-little_Lq):.2e}")
print(f"L       = {L:.8f}   | λ·W  = {little_L:.8f}   | dif = {abs(L-little_L):.2e}")
print("--- Variante sin espera ---")
print(f"L       = {L_sin:.8f}   | λ_ef·W = {little_L_sin:.8f}   | dif = {abs(L_sin-little_L_sin):.2e}")

# Error que se cometería usando λ en lugar de λ_ef en la variante sin espera
L_mal = lam * W_sin_d             # = a  (ignora que los bloqueados nunca entran)
print("\n--- Error al usar λ en vez de λ_ef (sin espera) ---")
print(f"λ·W = {L_mal:.4f}  vs  L correcto = {L_sin:.4f}   → sobreestima L en {(L_mal/L_sin - 1)*100:.2f} % (factor 1/(1-B) = {1/(1-B_ca):.4f})")
assert abs(Lq-little_Lq) < 1e-9 and abs(L-little_L) < 1e-9 and abs(L_sin-little_L_sin) < 1e-9


--- Variante con espera ---
Lq      = 2.28078714   | λ·Wq = 2.28078714   | dif = 0.00e+00
L       = 16.68078714   | λ·W  = 16.68078714   | dif = 0.00e+00
--- Variante sin espera ---
L       = 13.00720994   | λ_ef·W = 13.00720994   | dif = 0.00e+00

--- Error al usar λ en vez de λ_ef (sin espera) ---
λ·W = 14.4000  vs  L correcto = 13.0072   → sobreestima L en 10.71 % (factor 1/(1-B) = 1.1071)


In [6]:
# ---------- Resumen Task 1.1 ----------
print(f"{'Parámetros del grupo 4':-^62}")
print(f"λ = {lam:.1f}/día | c = {c} | a = {a:.1f} | ρ = {rho:.4f} | k_off = {k_off}")
print(f"{'Variante con espera (M/M/c)':-^62}")
print(f"C(c,a)={C_ca:.4f}  Lq={Lq:.4f}  Wq={Wq_h:.3f} h  W={W_h:.3f} h  L={L:.4f}")
print(f"{'Variante sin espera (M/M/c/c)':-^62}")
print(f"B(c,a)={B_ca:.4f}  λ_ef={lam_ef:.4f}/día  camas ocupadas={L_sin:.4f}")


--------------------Parámetros del grupo 4--------------------
λ = 3.6/día | c = 17 | a = 14.4 | ρ = 0.8471 | k_off = 6
-----------------Variante con espera (M/M/c)------------------
C(c,a)=0.4118  Lq=2.2808  Wq=15.205 h  W=111.205 h  L=16.6808
----------------Variante sin espera (M/M/c/c)-----------------
B(c,a)=0.0967  λ_ef=3.2518/día  camas ocupadas=13.0072


## Task 1.2 y 1.3 — Simulador de la IA, crítica y corrección

El código **original** que entregó la IA está en `task_1_2_ia.py` y **no se modifica** (anexo del Task 1.2). Aquí se audita con los siete puntos del Task 1.3 y se entrega la versión corregida.

Cada cambio del código corregido está marcado con `[CORRECCIÓN (n)]`.

### Código corregido (`task_1_3_corregido.py`)

In [ ]:
"""Simulador de eventos discretos del hospital (Task 1.3 – versión corregida).

Parte del código original de la IA (task_1_2_ia.py, que NO se modifica). El motor de eventos
ya era correcto (pasó las pruebas de respuesta conocida); lo que se corrige es la capa
estadística. Cada cambio está marcado con  [CORRECCIÓN (n)]  donde n es el punto de la
lista de auditoría (i)–(vii) del Task 1.3.

Modelo (visto en clase):
  * Llegadas de Poisson de tasa lam  ->  tiempos entre llegadas ~ Exp(lam)
  * Estancias ~ Exp(mu)  (media 1/mu días)
  * c camas.  waiting=True  -> M/M/c   (el paciente espera en cola si no hay cama)
              waiting=False -> M/M/c/c (el paciente se desvía y se pierde)
Todas las unidades de tiempo son DÍAS (las horas se obtienen multiplicando por 24).
"""
import heapq
import math
import random
from collections import deque

from scipy import stats


def simulate_hospital(lam, mu, c, horizon, warmup, seed, waiting=True, sample_every=None):
    """Una réplica. Devuelve las métricas de la ventana de observación [warmup, horizon].

    sample_every: si se da (en días), también devuelve la trayectoria (t, en_cola, en_sistema)
    muestreada desde t = 0, para justificar el calentamiento con un promedio móvil.
    """
    if lam <= 0 or mu <= 0 or c < 1 or not (0 <= warmup < horizon):
        raise ValueError('Parametros invalidos')

    rng = random.Random(seed)
    events = []          # lista de eventos futuros (heap ordenado por tiempo)
    counter = 0          # desempata eventos con el mismo tiempo

    def schedule(t, kind):
        nonlocal counter
        heapq.heappush(events, (t, counter, kind))
        counter += 1

    schedule(rng.expovariate(lam), 'arrival')
    t_prev = 0.0
    busy = 0                       # camas ocupadas
    queue = deque()                # (t_llegada, medido) de quienes esperan
    area_system = 0.0              # integral de N(t) dt   (N = pacientes en el sistema)
    area_queue = 0.0               # integral de Nq(t) dt  (Nq = pacientes en cola)
    area_busy = 0.0                # integral de camas_ocupadas(t) dt
    arrivals = 0
    blocked = 0
    waited = 0
    admitted = 0
    wait_sum = 0.0
    wait_count = 0
    trayectoria = []
    next_sample = 0.0

    while events:
        t, _, kind = heapq.heappop(events)
        end = min(t, horizon)
        left = max(t_prev, warmup)
        # (i) Promedio PONDERADO POR TIEMPO: L = (1/T) ∫ N(t) dt. El estado vigente se
        # multiplica por la duración del tramo [left, end]; solo cuenta la ventana medida.
        if end > left:
            duration = end - left
            area_system += (busy + len(queue)) * duration
            area_queue += len(queue) * duration
            area_busy += busy * duration
        # Muestreo de la trayectoria (incluye el calentamiento) para el promedio móvil.
        if sample_every:
            while next_sample <= end:
                trayectoria.append((next_sample, len(queue), busy + len(queue)))
                next_sample += sample_every
        t_prev = end

        # [CORRECCIÓN (vi)] El original hacía `break` al pasar el horizonte y descartaba a los
        # pacientes que seguían en cola: sus esperas largas nunca entraban en Wq (censura) y
        # P(esperar) y Wq tenían denominadores distintos. Ahora no se aceptan llegadas nuevas
        # después del horizonte, pero se deja terminar a los que ya llegaron, de modo que
        # TODO paciente medido tiene su espera completa. Las áreas siguen cortadas en `horizon`.
        if kind == 'arrival' and t > horizon:
            continue

        if kind == 'arrival':
            schedule(t + rng.expovariate(lam), 'arrival')
            measured = t >= warmup          # se mide por hora de LLEGADA (después del calentamiento)
            if measured:
                arrivals += 1
            if busy < c:
                busy += 1
                if measured:
                    admitted += 1
                    wait_count += 1         # espera 0 (entra directo): también cuenta para Wq
                schedule(t + rng.expovariate(mu), 'departure')
            elif waiting:
                queue.append((t, measured))
                if measured:
                    waited += 1             # P(esperar) = fracción que llega y no halla cama
                    admitted += 1
            else:
                if measured:
                    blocked += 1            # M/M/c/c: se pierde (B(c,a) en teoría)
        else:
            if queue:
                # (vii) Wq se mide desde la LLEGADA (t_llegada), no desde el inicio del servicio.
                t_llegada, measured = queue.popleft()
                if measured:
                    wait_count += 1
                    wait_sum += t - t_llegada
                schedule(t + rng.expovariate(mu), 'departure')
            else:
                busy -= 1

    observed = horizon - warmup
    nan = float('nan')
    resultado = {
        'L': area_system / observed,
        'Lq': area_queue / observed,
        # [CORRECCIÓN] sin datos se devuelve NaN, no 0.0 (un 0.0 inventado se vería como válido).
        'Wq_days': wait_sum / wait_count if wait_count else nan,
        'Wq_hours': 24 * wait_sum / wait_count if wait_count else nan,
        'utilization': area_busy / (c * observed),
        'prob_wait': waited / arrivals if arrivals else nan,
        'prob_block': blocked / arrivals if arrivals else nan,
        # [CORRECCIÓN (v)] tasa EFECTIVA de ingresos: λ_ef = λ(1 - B). Es la que va en la
        # ley de Little (L = λ_ef W); usar λ sobreestimaría L en la variante sin espera.
        'lambda_ef': admitted / observed,
        'arrivals': arrivals,
        'admitted': admitted,
        'blocked': blocked,
    }
    if sample_every:
        resultado['trayectoria'] = trayectoria
    return resultado


METRICAS = ('L', 'Lq', 'Wq_hours', 'utilization', 'prob_wait', 'prob_block', 'lambda_ef')


def simulate_replicas(n, master_seed, confidence=0.95, **params):
    """[CORRECCIÓN (ii) y (iv)] n réplicas independientes y IC por réplica.

    (iv) Las semillas se derivan de UN generador maestro (getrandbits), así son distintas y
         reproducibles; no se reutiliza ni se encadena la misma semilla.
    (ii) La unidad de muestreo es la RÉPLICA: cada réplica aporta un solo número por métrica
         (el promedio de toda su corrida). Los pacientes dentro de una réplica están
         correlacionados, por lo que NO se usan como muestra independiente.
         IC:  x̄ ± t_{1-α/2, n-1} · s/√n
    """
    maestro = random.Random(master_seed)
    seeds = []
    while len(seeds) < n:
        s = maestro.getrandbits(63)
        if s not in seeds:
            seeds.append(s)
    corridas = [simulate_hospital(seed=s, **params) for s in seeds]

    tcrit = stats.t.ppf(1 - (1 - confidence) / 2, n - 1)
    resumen = {}
    for m in METRICAS:
        xs = [r[m] for r in corridas if not math.isnan(r[m])]
        if len(xs) < 2:
            resumen[m] = (float('nan'), float('nan'), float('nan'))
            continue
        media = sum(xs) / len(xs)
        s = math.sqrt(sum((x - media) ** 2 for x in xs) / (len(xs) - 1))
        resumen[m] = (media, tcrit * s / math.sqrt(len(xs)), s)   # (media, semi-ancho, desv. est.)
    return resumen, corridas



### Pruebas con respuesta conocida

In [ ]:
"""Pruebas de auditoría del Task 1.3: se corren contra el código ORIGINAL de la IA (sin tocarlo)
y contra la versión corregida. Cada prueba tiene una respuesta conocida calculable a mano."""
import math
import statistics as st
from unittest import mock

import types
import task_1_2_ia as original   # código ORIGINAL de la IA, sin modificar (archivo task_1_2_ia.py)
# El módulo corregido es el de la celda anterior; se expone con la misma interfaz que usan las pruebas
corregido = types.SimpleNamespace(random=random, simulate_hospital=simulate_hospital, simulate_replicas=simulate_replicas)


class RngDeterminista:
    """Sustituye a random.Random: expovariate(tasa) = 1/tasa exactamente (sin azar).
    Convierte el simulador en un sistema D/D/c con resultados calculables a mano."""
    def __init__(self, seed): pass
    def expovariate(self, tasa): return 1.0 / tasa


def correr_det(modulo, **kw):
    with mock.patch.object(modulo.random, 'Random', RngDeterminista):
        return modulo.simulate_hospital(**kw)


def ok(cond): return 'OK ' if cond else 'FALLA'


print('=' * 78)
print('PRUEBA (i) D/D/1: llegada cada 3 d, estancia 1 d, 1 cama  ->  L = 1/3 (a mano)')
print('  Un promedio sobre eventos daría 0.5 (estados 1,0,1,0,...); el ponderado por tiempo, 1/3.')
for nombre, mod in (('original', original), ('corregido', corregido)):
    r = correr_det(mod, lam=1/3, mu=1.0, c=1, horizon=300, warmup=30, seed=0)
    print(f"  {nombre:10s} L={r['L']:.6f} util={r['utilization']:.6f}  esperado 0.333333  [{ok(abs(r['L']-1/3)<1e-9)}]")

print('=' * 78)
print('PRUEBAS (vi) y (vii) D/D/1 sobrecargado: llegada cada 1 d, estancia 1.5 d, 1 cama, T = 10 d')
print('  Espera del paciente n (desde su LLEGADA) = 0.5(n-1) d  ->  los 10 pacientes: 0, 0.5, ..., 4.5')
print('  Wq verdadero de los 10 = 2.25 d; los primeros 7 que alcanzan a entrar a servicio: 1.5 d.')
for nombre, mod in (('original', original), ('corregido', corregido)):
    r = correr_det(mod, lam=1.0, mu=1/1.5, c=1, horizon=10, warmup=0, seed=0)
    print(f"  {nombre:10s} llegadas={r['arrivals']} atendidos={r['admitted']} "
          f"P(esperar)={r['prob_wait']:.2f} Wq={r['Wq_days']:.4f} d")
print('  -> original: Wq=1.5 promedia solo 7 de 10 (descarta a los 3 que siguen en cola); '
      'P(esperar)=0.9 sí los cuenta.')
print('     Que 1.5 = promedio(0..3) confirma que mide desde la LLEGADA  [punto (vii) cumple].')

print('=' * 78)
print('PRUEBA (iv) reproducibilidad: misma semilla vs semilla distinta (original)')
p = dict(lam=3.6, mu=0.25, c=17, horizon=1000, warmup=200)
a = original.simulate_hospital(seed=42, **p); b = original.simulate_hospital(seed=42, **p)
d = original.simulate_hospital(seed=43, **p)
print(f"  misma semilla idéntica: {a == b}   semilla distinta idéntica: {a == d}")
print('  -> si se reusa la semilla, las "réplicas" serían copias y el IC tendría ancho 0.')

print('=' * 78)
print('PRUEBA (ii) variabilidad entre réplicas del ORIGINAL (30 semillas, grupo 4, con espera)')
lqs = [original.simulate_hospital(seed=1000 + i, waiting=True, **p)['Lq'] for i in range(30)]
print(f"  Lq por corrida: mín={min(lqs):.3f} máx={max(lqs):.3f} media={st.mean(lqs):.3f}  "
      f"(la corrida de la IA con semilla 42 dio {a['Lq']:.3f}; teoría 2.281)")
print('  -> una sola corrida no dice si 1.12 es error o azar; el original no entrega ningún IC.')

print('=' * 78)
print('PRUEBA (v) M/M/1/1 con lam=1, mu=1 (a=1): B=a/(1+a)=0.5, lambda_ef=0.5, L=0.5')
r = original.simulate_hospital(lam=1, mu=1, c=1, horizon=20000, warmup=200, seed=7, waiting=False)
lam_ef = r['admitted'] / (20000 - 200)
print(f"  original: L={r['L']:.4f}  lambda_ef(admitted/T)={lam_ef:.4f}  "
      f"claves devueltas incluyen 'lambda_ef': {'lambda_ef' in r}")
print(f"  Little con lambda_ef: lam_ef*W = {lam_ef:.4f}*1 = {lam_ef:.4f} ≈ L   |  con lambda: 1*1 = 1.0000 ≠ L")

print('=' * 78)
print('PRUEBA (iii) calentamiento: trayectoria media del número en cola desde un sistema VACÍO')
print('  (corregido, 100 réplicas, grupo 4 con espera; promedio de Nq por ventanas de 25 días)')
tray = [corregido.simulate_hospital(3.6, 0.25, 17, 400, 0, 5000 + i, True, sample_every=1.0)['trayectoria']
        for i in range(100)]
for ini in range(0, 400, 25):
    vals = [nq for t in tray for (tt, nq, _) in t if ini <= tt < ini + 25 for nq in [nq]]
    print(f"  días {ini:3d}-{ini+25:3d}: Lq medio = {st.mean(vals):.3f}")
print('  (teoría estacionaria: Lq = 2.281)')

print('=' * 78)
print('VALIDACIÓN del corregido con respuestas conocidas (30 réplicas, IC 95 %)')
res, _ = corregido.simulate_replicas(30, 11, lam=1, mu=1, c=1, horizon=5000, warmup=100, waiting=False)
m, h, _ = res['prob_block']; print(f"  M/M/1/1 a=1  P(bloqueo) = {m:.4f} ± {h:.4f}  teoría 0.5  [{ok(abs(m-0.5)<=h)}]")
m, h, _ = res['L'];          print(f"  M/M/1/1 a=1  L          = {m:.4f} ± {h:.4f}  teoría 0.5  [{ok(abs(m-0.5)<=h)}]")
res, _ = corregido.simulate_replicas(30, 12, lam=0.5, mu=1, c=1, horizon=5000, warmup=100, waiting=True)
for k, t in (('L', 1.0), ('Lq', 0.5), ('Wq_hours', 24.0), ('prob_wait', 0.5), ('utilization', 0.5)):
    m, h, _ = res[k]
    print(f"  M/M/1 ρ=0.5  {k:11s}= {m:.4f} ± {h:.4f}  teoría {t}  [{ok(abs(m-t)<=h)}]")
